In [2]:
import pandas as pd
import numpy as np
from autorep import Autorep
# from sklearn.model_selection import train_test_split
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import classification_report   



In [182]:
shanghai = pd.read_csv('disyllabic_wordlist.csv')
shanghai_test = shanghai[['word', 'tone']].sample(frac=0.1, random_state=1).reset_index(drop=True)


In [186]:


tone_map = {
    "4": "H",
    "5": "H",
    "3": "M",
    "1": "L",
    "2": "L",
}

tone_map2 = {
    "T1": "HL",
    "T2": "LH",
    "T3": "LH",
    "T4": "LH",
    "T5": "LH"
}

feature_system = {
    "voc": {
        "a", "i", "u", "o", "e", "y", "ø", "ɑ", "ɔ", "ə", "ɤ", "ɪ", "ɿ","ɛ"
    },
    "son": {
        "l", "m", "n", "ɲ", "ŋ", "ȵ"
    },
    "cons": {
        "b", "d", "f", "h", "k", "l", "m", "n", "p", "s", "t", "v", "z",
        "ɲ", "ȵ", "ɕ", "ŋ", "ɡ", "ɦ", "ʑ", "ʔ", "ʥ", "ʦ", "ʨ","ts", "dz", "tʃ", "dʒ", "tɕ", "th"
    },
    "cg": {
        "ʔ", "ɦ"  # consonantal glottalization
    },
    "+voi": {
        "b", "d", "ɡ", "z", "v", "l", "j", "ɦ", "ʥ"
    },
    "-voi": {
        "f", "h", "k", "p", "s", "t", "ʦ", "ʨ","ts","tʃ", "tɕ", "th"
    },
}


MULTI_SEGMENTS = {"ts", "dz", "tʃ", "dʒ", "tɕ", "th"}  # define your multi-character segments
def segmentize(word):
    """Split affricates into two consonants, treating as one"""
    segments = []
    i = 0
    while i < len(word):
        # Check if two-character combination matches a multi-segment
        if word[i:i+2] in MULTI_SEGMENTS:
            segments.append(word[i:i+2])
            i += 2
        else:
            segments.append(word[i])
            i += 1
    return segments


            
def get_tone(seq):
    """get tones from T1-T5 or 1-5 notation to H/M/L or HL/LH"""
    if seq in tone_map2:
        return tone_map2[seq]

# === Feature extraction === # this could be provided by the users

def get_feature(word):
    syllables = word.strip().split('.')
    word_features = []

    for syl in syllables:
        # print(syl,len(segmentize(syl)))
        
        for ch in segmentize(syl):
            features = [key for key, segs in feature_system.items() if ch in segs] or [None]
    
            word_features.append(features)

    return word_features

shanghai_test.loc[:, 'features'] = shanghai_test['word'].apply(get_feature)


shanghai_test


,word,tone,features
0,tsz.tɛ,T2-low,"[[cons, -voi], [cons, +voi], [cons, -voi], [voc]]"
1,thɔ.sɛ,T1-high,"[[cons, -voi], [voc], [cons, -voi], [voc]]"
2,fɛ.wɛ,T2,"[[cons, -voi], [voc], [None], [voc]]"
3,ɕi.pɛ,T2-high,"[[cons], [voc], [cons, -voi], [voc]]"
4,sɛ.fɛ,T1-low,"[[cons, -voi], [voc], [cons, -voi], [voc]]"
5,tɕiɔ.zɛ,T3-high,"[[cons, -voi], [voc], [voc], [cons, +voi], [voc]]"
6,zu.faʔ,T4-high,"[[cons, +voi], [voc], [cons, -voi], [voc], [co..."
7,tɕi.tɛ,T1-high,"[[cons, -voi], [voc], [cons, -voi], [voc]]"
8,ku.pɛ,T1-low,"[[cons, -voi], [voc], [cons, -voi], [voc]]"


In [187]:
shanghai_test = shanghai_test[
    shanghai_test["features"].apply(
        lambda feats: any("voc" in f for f in feats) if isinstance(feats, list) else False
    )
]
shanghai_test.to_csv('disyllabic_wordlist_filtered.csv', index=False)


In [193]:
test = pd.read_csv('disyllabic_wordlist_filtered.csv')

In [194]:
test

,word,tone,features
0,thɔ.sɛ,T1-high,"[['cons', '-voi'], ['voc'], ['cons', '-voi'], ..."
1,fɛ.wɛ,T2,"[['cons', '-voi'], ['voc'], [None], ['voc']]"
2,ɕi.pɛ,T2-high,"[['cons'], ['voc'], ['cons', '-voi'], ['voc']]"
3,sɛ.fɛ,T1-low,"[['cons', '-voi'], ['voc'], ['cons', '-voi'], ..."
4,tɕiɔ.zɛ,T3-high,"[['cons', '-voi'], ['voc'], ['voc'], ['cons', ..."
5,zu.faʔ,T4-high,"[['cons', '+voi'], ['voc'], ['cons', '-voi'], ..."
6,tɕi.tɛ,T1-high,"[['cons', '-voi'], ['voc'], ['cons', '-voi'], ..."
7,ku.pɛ,T1-low,"[['cons', '-voi'], ['voc'], ['cons', '-voi'], ..."


In [ ]:
# create three slots for each syllable


def get_tone(word):
    syllables = word.strip().split('.')
    word_features = []

    for syl in syllables:
        syl_features = []
        for ch in segmentize(syl):
            features = [key for key, segs in feature_system.items() if ch in segs] or [None]
            syl_features.append({ch: features})
        word_features.append(syl_features)

    return word_features